In [ ]:
# Imports
import pandas as pd
import numpy as np
import ast
import xgboost as xgb
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from collections import Counter

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Constants
file_path = 'data.csv'
columns = [
    # 'id', 'name',
    'host_is_superhost',
    'latitude', 'longitude', 'room_type', 'accommodates',
    'bathrooms', 'bathrooms_text', 'bedrooms', 'beds',
    'amenities', 'price', 'review_scores_rating', 'availability_365', 'neighbourhood_cleansed'
]
# Source: https://www.latlong.net/place/london-uk-14153.html
london_center_lat = 51.509865
london_center_lon = -0.118092

# Random seeds — we train and evaluate the model once per seed, then average
# the results for a more reliable performance estimate than a single run.
seeds = [1, 7, 42, 67, 99]

In [ ]:
# Load the raw dataset
df = pd.read_csv(file_path)

In [ ]:
# Show the full dataframe without truncation — useful for initial exploration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

print(df.columns.tolist())
print(df.head())

In [ ]:
# Keep only the relevant columns and drop rows that have any missing values
df = df[columns].dropna()

In [ ]:
# Explore the neighbourhood column before encoding it
print(df['neighbourhood_cleansed'].nunique())       # how many unique neighbourhoods?
print(df['neighbourhood_cleansed'].value_counts().head(10))

In [ ]:
# One-hot encode the neighbourhood column:
# Each neighbourhood becomes its own 0/1 column (e.g. nb_Westminster = 1 if the listing is there).
nb_dummies = pd.get_dummies(df['neighbourhood_cleansed'], prefix='nb', dtype=int)
df = pd.concat([df, nb_dummies], axis=1)
df = df.drop(columns='neighbourhood_cleansed')

In [ ]:
def calculate_haversine_distance(lat_series, lon_series, center_lat, center_lon):
    """
    Calculate the straight-line distance (in meters) between each listing and a fixed point.

    Uses the Haversine formula, which correctly accounts for the Earth's curvature.
    The function is vectorized — it processes the entire column at once, which is very fast.
    """
    R = 6371000.0  # Earth's radius in meters

    # Convert degrees → radians (required for Python's trig functions)
    phi1         = np.radians(lat_series)
    phi2         = np.radians(center_lat)
    delta_phi    = np.radians(center_lat - lat_series)
    delta_lambda = np.radians(center_lon - lon_series)

    # Haversine formula
    a = np.sin(delta_phi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

In [ ]:
# Replace raw lat/lon with a single distance-to-center feature (in meters)
df['distance_to_center_m'] = calculate_haversine_distance(
    df['latitude'], df['longitude'],
    london_center_lat, london_center_lon
).round()

df = df.drop(columns=['latitude', 'longitude'])

In [ ]:
print(df.dtypes)

In [ ]:
# Convert 't'/'f' text values to 1/0 integers (1 = is a superhost)
df['host_is_superhost'] = (df['host_is_superhost'] == 't').astype(int)

In [ ]:
# One-hot encode room type into four separate 0/1 columns
print(df['room_type'].unique())
df["is_private_room"] = (df["room_type"] == "Private room").astype(int)
df["is_entire_home"]  = (df["room_type"] == "Entire home/apt").astype(int)
df["is_hotel_room"]   = (df["room_type"] == "Hotel room").astype(int)
df["is_shared_room"]  = (df["room_type"] == "Shared room").astype(int)

df = df.drop(columns=['room_type'])

In [ ]:
# Remove rare room types — hotel (41 listings) and shared (111 listings) are too few to learn from
df = df[(df['is_hotel_room'] == False) & (df['is_shared_room'] == False)]
df = df.drop(columns=['is_hotel_room', 'is_shared_room'])

In [ ]:
# Clean the price column: strip "$" and "," characters, then convert to a numeric float
df['price_dollar'] = (df['price']
                      .str.replace("$", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .str.replace(".00", "", regex=False)
                      .astype(float))
df = df.drop(columns=['price'])

In [ ]:
# Create a 1/0 flag: 1 if the bathroom is shared, 0 if private
df['is_shared_bath'] = df['bathrooms_text'].str.contains('shared', case=False, na=False).astype(int)
df = df.drop(columns='bathrooms_text')

In [ ]:
# Drop amenities column (not used in this model version)
df = df.drop(columns='amenities')

# Remove extreme price outliers — listings above $1000/night are likely data errors
df = df[df['price_dollar'] < 1000]

In [ ]:
df

In [ ]:
# ── Prepare Features and Target ───────────────────────────────────────────────
X = df.drop(columns='price_dollar')
y = np.log1p(df['price_dollar'])  # log-transform: makes the skewed price distribution easier to predict

# ── Hyperparameter Tuning with Optuna (using first seed) ─────────────────────
# Split data: 80% for training, 20% held out for final testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seeds[0])

# Standardize features: subtract mean, divide by std.
# Important: fit ONLY on training data to avoid leaking test information.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Further split training data into train/validation for Optuna's early stopping
X_tr, X_val, y_tr, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=seeds[0])

def objective(trial):
    """
    Optuna calls this function once per trial.
    It trains XGBoost with a suggested set of hyperparameters and
    returns the validation RMSE — Optuna tries to minimize this value.
    """
    params = {
        'n_estimators':          1000,
        'early_stopping_rounds': 50,        # stop early if val-RMSE stops improving
        'eval_metric':           'rmse',
        'learning_rate':         trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':             trial.suggest_int('max_depth', 3, 10),
        'subsample':             trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':      trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':      trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':             trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':            trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'random_state':          seeds[0],
        'verbosity':             0,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict(X_val)
    return root_mean_squared_error(y_val, preds)

# Run hyperparameter search (increase n_trials for better results)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print(f"Best Val RMSE : {study.best_value:.4f}")
print(f"Best Params   : {study.best_params}")

# ── Evaluate Across Multiple Seeds ────────────────────────────────────────────
# We re-train the model with the best hyperparameters on different random splits.
# Averaging RMSE across all seeds gives a more reliable estimate of real-world
# performance than a single train/test split.
best_params = study.best_params | {
    'n_estimators':          1000,
    'early_stopping_rounds': 50,
    'eval_metric':           'rmse',
    'verbosity':             0,
}

rmse_scores = []

for seed in seeds:
    # Create a fresh train/test split using this seed
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y, test_size=0.2, random_state=seed)

    # Fit a new scaler on this seed's training data only (prevents data leakage)
    scaler_s = StandardScaler()
    X_train_s_scaled = scaler_s.fit_transform(X_train_s)
    X_test_s_scaled  = scaler_s.transform(X_test_s)

    # Validation split for early stopping
    X_tr_s, X_val_s, y_tr_s, y_val_s = train_test_split(
        X_train_s_scaled, y_train_s, test_size=0.2, random_state=seed
    )

    # Train with the best hyperparameters found by Optuna
    model = xgb.XGBRegressor(**best_params, random_state=seed)
    model.fit(X_tr_s, y_tr_s, eval_set=[(X_val_s, y_val_s)], verbose=False)

    # Convert log-predictions back to dollar values, then compute RMSE
    y_pred_s = np.expm1(model.predict(X_test_s_scaled))
    y_true_s = np.expm1(y_test_s)
    rmse_s   = root_mean_squared_error(y_true_s, y_pred_s)

    rmse_scores.append(rmse_s)
    print(f"Seed {seed:>3}: Test RMSE = ${rmse_s:.2f}")

print(f"\nMean RMSE : ${np.mean(rmse_scores):.2f}  ±  ${np.std(rmse_scores):.2f}")

# Keep the last trained model for the feature importance plots below
final_model = model

In [ ]:
importances = pd.Series(final_model.feature_importances_, index=X.columns)
importances.sort_values().tail(20).plot(kind='barh')
plt.tight_layout()
plt.show()

In [ ]:
# Plot feature importance by "gain" — how much each feature reduces prediction error on average.
# This is more informative than the default "weight" metric (which just counts splits).
booster = final_model.get_booster()
booster.feature_names = list(X.columns)

importance_dict = booster.get_score(importance_type='gain')
importances = pd.Series(importance_dict).sort_values()

importances.tail(20).plot(kind='barh', figsize=(10, 8))
plt.xlabel('Importance (Gain)')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
def safe_parse(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

# Häufigkeiten zählen
all_amenities = df["amenities"].apply(safe_parse).explode()
counts = Counter(all_amenities)

# Top 50
top_n = 50
top_amenities = [amenity for amenity, count in counts.most_common(top_n)]
print(top_amenities)

# Nur Top-N im DataFrame behalten
top_set = set(top_amenities)
df["amenities_filtered"] = df["amenities"].apply(safe_parse).apply(
    lambda lst: [a for a in lst if a in top_set]
)